# GP01 — First GeoPython Workflow

Load → Inspect → Understand → Filter → Map → Validate → Export

Η λύση κρατά το source `points`, δημιουργεί `gdf` με `Point` geometry και EPSG:4326, ελέγχει το subset πριν από το map και ξαναδιαβάζει το export. Κάθε στάδιο απαντά σε διαφορετική QA ερώτηση.

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path

In [ ]:
csv_path = "data/gp01/survey_points.csv"
points = pd.read_csv(csv_path)
print("Γραμμές:", len(points))
print("Στήλες:", list(points.columns))
print("Types:\n", points.dtypes)
print("Missing:\n", points.isna().sum())
points.head()

In [ ]:
geometry = [Point(xy) for xy in zip(points["longitude"], points["latitude"]) ]
gdf = gpd.GeoDataFrame(points, geometry=geometry, crs="EPSG:4326")
print("CRS:", gdf.crs)
gdf

In [ ]:
selected = gdf[
    (gdf["elevation_m"] > 70) &
    (gdf["status"] == "checked")
] .copy()
print("Επιλεγμένες γραμμές:", len(selected))
selected

In [ ]:
ax = gdf.plot(figsize=(8, 6), color="lightgray", edgecolor="black")
selected.plot(ax=ax, color="red", markersize=60)

In [ ]:
assert len(gdf) > 0
assert gdf.geometry.notna().all()
assert gdf.geometry.geom_type.eq("Point").all()
assert gdf.crs is not None
assert gdf["longitude"].between(-180, 180).all()
assert gdf["latitude"].between(-90, 90).all()
assert len(selected) > 0
assert (selected["elevation_m"] > 70).all()
assert (selected["status"] == "checked").all()
print("Όλα τα QA checks πέρασαν.")

In [ ]:
output_dir = Path("outputs/gp01")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "high_checked_points.geojson"
selected.to_file(output_path, driver="GeoJSON")
exported = gpd.read_file(output_path)
print("Αποθηκεύτηκε:", output_path)
print("Γραμμές export:", len(exported))
print("CRS export:", exported.crs)

## Mini Challenge — Ενδεικτική λύση

In [ ]:
water_low = gdf[
    (gdf["category"] == "water") &
    (gdf["elevation_m"] < 65)
] .copy()
assert len(water_low) > 0
assert water_low.geometry.notna().all()
challenge_path = output_dir / "water_low_points.geojson"
water_low.to_file(challenge_path, driver="GeoJSON")
print("Challenge export:", challenge_path)

## Τι πρέπει να κρατήσεις

Ένα επαναλήψιμο GeoPython workflow φορτώνει, επιθεωρεί, κατανοεί, φιλτράρει, χαρτογραφεί, ελέγχει και εξάγει.